# Week 2 Day 3 — LangGraph

Stateful, multi-step & cyclical agent workflows using `StateGraph`, conditional edges, human-in-the-loop interrupts, and persistence with `MemorySaver`.

## Task 1: Graph concepts & state design

LangGraph core building blocks:
- **StateGraph**: graph of nodes with shared State
- **Nodes**: functions that take State and return partial updates
- **Edges**: transitions between nodes
- **Conditional edges**: routers choose the next node

### State schema (shared object)
We model a shopping/recommendation workflow with a single `ShoppingState` TypedDict. Key fields: `product_a`, `product_b`, `retrieved`, `draft`, `quality_score`, `retries`, `human_approved`, `formatted`, and `trace` (for debugging).

### Graph diagram (final workflow)

See also `graph_diagram.md`.

```mermaid
flowchart TD
    A([START]) --> P[plan]
    P --> R[retrieve]
    R --> G[generate]
    G --> C[critique]
    C -->|quality < threshold AND retries < max| G
    C -->|acceptable| H[human_review (interrupt_before)]
    H -->|approved| F[format]
    H -->|rejected| G
    F --> E([END])
```

In [1]:
from langgraph_agent import (
    build_linear_graph,
    build_conditional_graph,
    build_human_in_loop_graph,
    default_initial_state,
    get_default_checkpointer,
)

print("Imported LangGraph workflow helpers.")

Imported LangGraph workflow helpers.


## Task 2: Linear graph

Compile a simple graph: plan → retrieve → generate → format.

We print the `trace` list (it records what each node did).

In [2]:
graph, checkpointer = build_linear_graph()
init = default_initial_state(
    request="compare",
    product_a="Wireless Mouse",
    product_b="Mechanical Keyboard",
    quality_threshold=0.8,
    max_retries=2,
)
cfg = {"configurable": {"thread_id": "day3-linear"}}

out = graph.invoke(init, cfg)
print("\n--- Linear graph output formatted ---")
print(out["formatted"][:300])
print("\n--- Trace tail ---")
print("\n".join((out.get("trace") or [])[-6:]))


--- Linear graph output formatted ---
FINAL RECOMMENDATION

Products: Wireless Mouse ($18.99) vs Mechanical Keyboard ($79.50)
Difference: $60.51
Recommendation: Wireless Mouse

Why: Compare Wireless Mouse vs Mechanical Keyboard for budget-conscious buying. The cheaper option is likely Wireless Mouse. Add more precise price details after

--- Trace tail ---
[plan_node] plan=['Retrieve catalog info for product A and B.', 'Draft a comparison and budget-friendly recommendation.', 'Critique draft quality; loop back to draft if needed.', 'Ask for human approval before formatting final response.']
[retrieve_node] retrieved={"a": {"product_id": "P001", "name": "Wireless Mouse", "price_usd": 18.99, "in_stock": true}, "b": {"product_id": "P002", "name": "Mechanical Keyboard", "price_usd": 79.5, "in_stock": true}}
[generate_node] retries=0 draft_len=164
[format_node] formatted_len=368


## Task 3: Conditional edges & cycles

The graph loops back from `critique` → `generate` if the heuristic `quality_score` is below a threshold. We also enforce `max_retries` to prevent infinite cycles.

In [3]:
cp = get_default_checkpointer()
graph_cond = build_conditional_graph(cp)
init = default_initial_state(
    request="compare",
    product_a="Wireless Mouse",
    product_b="Mechanical Keyboard",
    quality_threshold=0.8,
    max_retries=2,
)
cfg = {"configurable": {"thread_id": "day3-conditional"}}

out = graph_cond.invoke(init, cfg)
print("\n--- Conditional graph formatted ---")
print(out["formatted"][:260])
print("\n--- Trace (last 10 entries) ---")
print("\n".join((out.get("trace") or [])[-10:]))


--- Conditional graph formatted ---
FINAL RECOMMENDATION

Products: Wireless Mouse ($18.99) vs Mechanical Keyboard ($79.50)
Difference: $60.51
Recommendation: Wireless Mouse

Why: Budget comparison for Wireless Mouse vs Mechanical Keyboard.

- Wireless Mouse: $18.99
- Mechanical Keyboard: $79.50

--- Trace (last 10 entries) ---
[plan_node] plan=['Retrieve catalog info for product A and B.', 'Draft a comparison and budget-friendly recommendation.', 'Critique draft quality; loop back to draft if needed.', 'Ask for human approval before formatting final response.']
[retrieve_node] retrieved={"a": {"product_id": "P001", "name": "Wireless Mouse", "price_usd": 18.99, "in_stock": true}, "b": {"product_id": "P002", "name": "Mechanical Keyboard", "price_usd": 79.5, "in_stock": true}}
[generate_node] retries=0 draft_len=164
[critique_node] quality_score=0.6000000000000001 (threshold=0.8) pass=1 retries=0->1
[generate_node] retries=1 draft_len=183
[critique_node] quality_score=1.0 (threshold=0.8

Explain (2–3 sentences) why this loop-back control-flow is hard to express in plain AgentExecutor but natural in LangGraph:

When using a plain AgentExecutor, you typically implement one monolithic loop that mixes reasoning, tool usage, and control. With multiple routing decisions (critique → loop back; human approval → either finish or revise) the logic becomes harder to make explicit and to test. LangGraph makes those transitions first-class via conditional edges and a shared state that is easy to inspect and replay.

## Task 4: Human-in-the-loop & interrupts

We pause the graph right before `human_review` using `interrupt_before=["human_review"]`.

Then we simulate human approval/rejection by calling `graph.update_state(...)` and resuming.

In [4]:
cp = get_default_checkpointer()
graph_hil = build_human_in_loop_graph(cp)

# ---------- Run A: human approves ----------
thread_id = "day3-hil-approve"
cfg = {"configurable": {"thread_id": thread_id}}
init = default_initial_state(
    request="compare",
    product_a="Wireless Mouse",
    product_b="Mechanical Keyboard",
    quality_threshold=0.8,
    max_retries=3,
)

last = None
for state in graph_hil.stream(init, cfg, stream_mode="values"):
    last = state

print("\n--- Paused (approval run) trace tail ---")
print((last.get("trace") or [])[-3:])

# Simulate human approval
graph_hil.update_state(
    cfg,
    {"human_approved": True, "human_notes": "Approved by reviewer."},
    as_node="human_review",
)

outA = graph_hil.invoke(None, cfg, interrupt_before=[])
print("\n--- Final formatted (approved) ---")
print(outA["formatted"][:220])

# ---------- Run B: human rejects once then approves ----------
thread_id = "day3-hil-reject-then-approve"
cfg = {"configurable": {"thread_id": thread_id}}
init = default_initial_state(
    request="compare",
    product_a="Wireless Mouse",
    product_b="Mechanical Keyboard",
    quality_threshold=0.8,
    max_retries=3,
)

last = None
for state in graph_hil.stream(init, cfg, stream_mode="values"):
    last = state

print("\n--- Paused (rejection run #1) trace tail ---")
print((last.get("trace") or [])[-3:])

# Reject
graph_hil.update_state(
    cfg,
    {"human_approved": False, "human_notes": "Reject: add price numbers."},
    as_node="human_review",
)

# Resume until next interrupt point
last2 = None
for state in graph_hil.stream(None, cfg, stream_mode="values"):
    last2 = state

print("\n--- Paused (rejection run #2) trace tail ---")
print((last2.get("trace") or [])[-3:])

# Approve
graph_hil.update_state(
    cfg,
    {"human_approved": True, "human_notes": "Now acceptable."},
    as_node="human_review",
)

outB = graph_hil.invoke(None, cfg, interrupt_before=[])
print("\n--- Final formatted (rejected then approved) ---")
print(outB["formatted"][:220])


--- Paused (approval run) trace tail ---
['[critique_node] quality_score=0.6000000000000001 (threshold=0.8) pass=1 retries=0->1', '[generate_node] retries=1 draft_len=183', '[critique_node] quality_score=1.0 (threshold=0.8) pass=2 retries=1->1']

--- Final formatted (approved) ---
FINAL RECOMMENDATION

Products: Wireless Mouse ($18.99) vs Mechanical Keyboard ($79.50)
Difference: $60.51
Recommendation: Wireless Mouse

Why: Budget comparison for Wireless Mouse vs Mechanical Keyboard.

- Wireless Mou

--- Paused (rejection run #1) trace tail ---
['[critique_node] quality_score=0.6000000000000001 (threshold=0.8) pass=1 retries=0->1', '[generate_node] retries=1 draft_len=183', '[critique_node] quality_score=1.0 (threshold=0.8) pass=2 retries=1->1']

--- Paused (rejection run #2) trace tail ---
['[critique_node] quality_score=1.0 (threshold=0.8) pass=2 retries=1->1', '[generate_node] retries=1 draft_len=183', '[critique_node] quality_score=1.0 (threshold=0.8) pass=3 retries=1->1']

--- Fina

## Task 5: Persistence & debugging

Checkpointer ki wajah se state history available hoti hai (same thread_id pe resuming). `get_state_history` se time-travel/debugging ka snapshot milta hai.

In [5]:
cfg = {"configurable": {"thread_id": "day3-hil-reject-then-approve"}}
state = graph_hil.get_state(cfg)
print("Current state keys:", list(state.values.keys()))

hist = list(graph_hil.get_state_history(cfg))
print("State history length:", len(hist))

print("Last state snapshot trace tail:")
last = hist[-1].values
print((last.get("trace") or [])[-5:])

Current state keys: ['request', 'product_a', 'product_b', 'plan', 'retrieved', 'draft', 'quality_score', 'retries', 'max_retries', 'quality_threshold', 'loop_passes', 'human_approved', 'human_notes', 'formatted', 'trace']
State history length: 13
Last state snapshot trace tail:
[]


## LangChain AgentExecutor vs LangGraph (short comparison)

- **AgentExecutor**: fast to assemble tool-calling loops, but complex routing/interrupt/retry logic mostly lives in custom code.
- **LangGraph**: explicit control-flow (nodes/edges/routers), easy cycles, deterministic state, built-in persistence and replay.

In production: use LangChain for simpler agents; use LangGraph when you need reliability, debuggability, and multi-step workflow control.